In [ ]:

# Assignment 4
# DDS-8555 Predictive Analysis
# Code-only notebook for ISLR Python:
#   Conceptual Question 3
#   Applied Question 8

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import dmatrix
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, SplineTransformer, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True


In [ ]:

# -----------------------------
# Conceptual Question 3
# -----------------------------
# I define the basis functions exactly as stated in the prompt.

def b1(x):
    return x

def b2(x):
    return ((x - 1) ** 2) * (x >= 1)

beta0 = 1
beta1 = 1
beta2 = -2

def y_hat(x):
    return beta0 + beta1 * b1(x) + beta2 * b2(x)

# I evaluate the fitted curve on a dense grid so I can visualize the piecewise form.
x = np.linspace(-2, 2, 401)
y = y_hat(x)

# I write the piecewise equations explicitly because the sketch is easier to verify this way.
# For x < 1:  y = 1 + x
# For x >= 1: y = 1 + x - 2(x - 1)^2 = -2x^2 + 5x - 1

# I compute the key quantities that matter for the sketch.
y_intercept = y_hat(0)
x_intercept_left = -1
vertex_x = 5 / 4
vertex_y = y_hat(vertex_x)
left_value_at_1 = 1 + 1
right_value_at_1 = -2 * (1 ** 2) + 5 * 1 - 1

left_slope = 1
right_slope_at_1 = 1  # derivative of -2x^2 + 5x - 1 at x=1 is -4(1) + 5 = 1

print("Piecewise fitted curve:")
print("For X < 1,  Ŷ = 1 + X")
print("For X >= 1, Ŷ = 1 + X - 2(X - 1)^2 = -2X^2 + 5X - 1")
print()
print(f"Y-intercept: (0, {y_intercept:.3f})")
print(f"X-intercept in the plotted range: ({x_intercept_left:.3f}, 0)")
print(f"Value from the left at X = 1: {left_value_at_1:.3f}")
print(f"Value from the right at X = 1: {right_value_at_1:.3f}")
print(f"Left slope at X = 1: {left_slope:.3f}")
print(f"Right slope at X = 1: {right_slope_at_1:.3f}")
print(f"Turning point on the quadratic portion: ({vertex_x:.3f}, {vertex_y:.3f})")

plt.figure()
plt.plot(x, y, linewidth=2, label="Estimated curve")
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.axvline(1, linestyle="--", linewidth=1, label="Knot at X = 1")

plt.scatter(
    [0, x_intercept_left, 1, vertex_x],
    [y_intercept, 0, y_hat(1), vertex_y],
    s=60,
    label="Key points"
)

plt.annotate("y-intercept (0, 1)", (0, y_intercept), xytext=(0.15, 1.25))
plt.annotate("x-intercept (-1, 0)", (x_intercept_left, 0), xytext=(-1.75, -0.5))
plt.annotate("continuous at x = 1", (1, y_hat(1)), xytext=(1.05, 2.2))
plt.annotate("local max (1.25, 2.125)", (vertex_x, vertex_y), xytext=(1.35, 2.4))

plt.xlim(-2, 2)
plt.ylim(min(y) - 0.5, max(y) + 0.5)
plt.xlabel("X")
plt.ylabel("Estimated Y")
plt.title("Conceptual Question 3: Estimated Curve")
plt.legend()
plt.show()


In [ ]:
# -----------------------------
# Applied Question 8
# -----------------------------
# I import the Auto data set directly from the ISLP library rather than from a local CSV file.
# This keeps the notebook aligned with the textbook workflow and avoids machine-specific file path issues.

try:
    from ISLP import load_data
except ImportError as exc:
    raise ImportError(
        "The ISLP package is required to load the Auto data set. "
        "Please run `%pip install ISLP` in a notebook cell, then restart the kernel and rerun the notebook."
    ) from exc

auto = load_data("Auto")
auto["origin"] = auto["origin"].astype("category")

print(auto.shape)
print(auto.dtypes)
auto.head()

In [ ]:

# I verify that the imported file is clean before fitting any models.

print(auto.isna().sum())
print()
print(auto.describe(include="all").T)


In [ ]:

# I start with horsepower because mpg and horsepower are known to have a visibly curved relationship.
# I fit four models:
#   1. simple linear regression
#   2. quadratic regression
#   3. cubic regression
#   4. cubic spline regression
#
# I compare them with 10-fold cross-validated RMSE.

X_hp = auto[["horsepower"]]
y_mpg = auto["mpg"]

cv = KFold(n_splits=10, shuffle=True, random_state=8555)

models_hp = {
    "Linear": Pipeline([
        ("model", LinearRegression())
    ]),
    "Quadratic": Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("model", LinearRegression())
    ]),
    "Cubic": Pipeline([
        ("poly", PolynomialFeatures(degree=3, include_bias=False)),
        ("model", LinearRegression())
    ]),
    "Cubic spline (df=6)": Pipeline([
        ("spline", SplineTransformer(degree=3, n_knots=5, include_bias=False)),
        ("model", LinearRegression())
    ])
}

hp_results = []
for name, model in models_hp.items():
    rmse = -cross_val_score(
        model,
        X_hp,
        y_mpg,
        scoring="neg_root_mean_squared_error",
        cv=cv
    ).mean()
    hp_results.append({"model": name, "cv_rmse": rmse})

hp_results = pd.DataFrame(hp_results).sort_values("cv_rmse").reset_index(drop=True)
hp_results


In [ ]:

# I generate fitted curves for the horsepower models so I can inspect the shape directly.

x_grid_hp = np.linspace(auto["horsepower"].min(), auto["horsepower"].max(), 400).reshape(-1, 1)

plt.figure()
plt.scatter(auto["horsepower"], auto["mpg"], alpha=0.5, label="Observed data")

for name, model in models_hp.items():
    model.fit(X_hp, y_mpg)
    y_grid = model.predict(x_grid_hp)
    plt.plot(x_grid_hp, y_grid, linewidth=2, label=name)

plt.xlabel("Horsepower")
plt.ylabel("MPG")
plt.title("Applied Question 8: MPG vs Horsepower")
plt.legend()
plt.show()


In [ ]:

# I also inspect weight because mpg often declines with weight in a way that is not perfectly linear.

X_wt = auto[["weight"]]

models_wt = {
    "Linear": Pipeline([
        ("model", LinearRegression())
    ]),
    "Quadratic": Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("model", LinearRegression())
    ]),
    "Cubic": Pipeline([
        ("poly", PolynomialFeatures(degree=3, include_bias=False)),
        ("model", LinearRegression())
    ]),
    "Cubic spline (df=6)": Pipeline([
        ("spline", SplineTransformer(degree=3, n_knots=5, include_bias=False)),
        ("model", LinearRegression())
    ])
}

wt_results = []
for name, model in models_wt.items():
    rmse = -cross_val_score(
        model,
        X_wt,
        y_mpg,
        scoring="neg_root_mean_squared_error",
        cv=cv
    ).mean()
    wt_results.append({"model": name, "cv_rmse": rmse})

wt_results = pd.DataFrame(wt_results).sort_values("cv_rmse").reset_index(drop=True)
wt_results


In [ ]:

# I draw the fitted weight curves.

x_grid_wt = np.linspace(auto["weight"].min(), auto["weight"].max(), 400).reshape(-1, 1)

plt.figure()
plt.scatter(auto["weight"], auto["mpg"], alpha=0.5, label="Observed data")

for name, model in models_wt.items():
    model.fit(X_wt, y_mpg)
    y_grid = model.predict(x_grid_wt)
    plt.plot(x_grid_wt, y_grid, linewidth=2, label=name)

plt.xlabel("Weight")
plt.ylabel("MPG")
plt.title("Applied Question 8: MPG vs Weight")
plt.legend()
plt.show()


In [ ]:

# I fit a broader additive-style model with spline terms for continuous predictors and a categorical term for origin.
# This gives me a multivariable nonlinear model I can compare against a more conventional linear baseline.

linear_formula = "mpg ~ horsepower + weight + displacement + acceleration + year + C(origin)"
spline_formula = (
    "mpg ~ bs(horsepower, df=6, degree=3) + "
    "bs(weight, df=6, degree=3) + "
    "bs(displacement, df=5, degree=3) + "
    "bs(acceleration, df=5, degree=3) + "
    "year + C(origin)"
)

linear_model = smf.ols(linear_formula, data=auto).fit()
spline_model = smf.ols(spline_formula, data=auto).fit()

comparison = pd.DataFrame({
    "model": ["Linear baseline", "Spline-based additive model"],
    "AIC": [linear_model.aic, spline_model.aic],
    "BIC": [linear_model.bic, spline_model.bic],
    "Adjusted_R2": [linear_model.rsquared_adj, spline_model.rsquared_adj]
})
comparison


In [ ]:

# I plot observed versus fitted values for the linear baseline and the spline-based model.

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(auto["mpg"], linear_model.fittedvalues, alpha=0.6)
axes[0].plot([auto["mpg"].min(), auto["mpg"].max()],
             [auto["mpg"].min(), auto["mpg"].max()],
             linewidth=2)
axes[0].set_title("Linear Baseline")
axes[0].set_xlabel("Observed MPG")
axes[0].set_ylabel("Fitted MPG")

axes[1].scatter(auto["mpg"], spline_model.fittedvalues, alpha=0.6)
axes[1].plot([auto["mpg"].min(), auto["mpg"].max()],
             [auto["mpg"].min(), auto["mpg"].max()],
             linewidth=2)
axes[1].set_title("Spline-Based Additive Model")
axes[1].set_xlabel("Observed MPG")
axes[1].set_ylabel("Fitted MPG")

plt.tight_layout()
plt.show()


In [ ]:

# I print a concise code-generated conclusion so the main findings are preserved inside the notebook itself.

best_hp = hp_results.iloc[0]
best_wt = wt_results.iloc[0]

print("Summary for Applied Question 8")
print("------------------------------")
print(f"Best horsepower model by 10-fold CV RMSE: {best_hp['model']} (RMSE = {best_hp['cv_rmse']:.3f})")
print(f"Best weight model by 10-fold CV RMSE: {best_wt['model']} (RMSE = {best_wt['cv_rmse']:.3f})")
print()
print("The fitted curves and the cross-validated error results indicate that the relationships")
print("between mpg and both horsepower and weight are not purely linear. The spline-based")
print("additive model also improves AIC, BIC, and adjusted R-squared relative to the linear")
print("baseline, which supports the conclusion that meaningful non-linearity is present in")
print("the Auto data set.")
